# Notebook 3: Comparación de Arquitecturas YOLO

**Trabajo Práctico – Visión por Computadora II**  

---

## Objetivos
1. Comparar YOLOv8n, YOLOv8m y YOLOv11n en mAP50, mAP50-95, Precision, Recall
2. Analizar el trade-off velocidad vs. precisión
3. Comparar métricas por clase (EPP presente vs. EPP ausente)
4. Justificar la elección del modelo óptimo para el sistema de monitoreo

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style="whitegrid")

# Cargar resultados
with open("../data/eval_results.json") as f:
    eval_results = json.load(f)

with open("../data/latency_results.json") as f:
    latency_results = json.load(f)

with open("../data/dataset_config.json") as f:
    ds_config = json.load(f)

DATASET_YAML = ds_config["dataset_yaml"]
MODELS_DIR = Path("../models")
RUNS_DIR   = Path("../runs")
DATA_DIR   = Path("../data")

print("Resultados cargados:")
print(json.dumps(eval_results, indent=2))

FileNotFoundError: [Errno 2] No such file or directory: '../data/eval_results.json'

## 1. Tabla Comparativa General

In [ ]:
# Combinar métricas y latencia
rows = []
model_info = {
    'YOLOv8n':  {'params_M': 3.2,  'flops_G': 8.7},
    'YOLOv8m':  {'params_M': 25.9, 'flops_G': 78.9},
    'YOLOv11n': {'params_M': 2.6,  'flops_G': 6.5},
}

for model_name in eval_results:
    row = {'Modelo': model_name}
    row.update(eval_results[model_name])
    if model_name in latency_results:
        row.update(latency_results[model_name])
    if model_name in model_info:
        row.update(model_info[model_name])
    rows.append(row)

df = pd.DataFrame(rows).set_index('Modelo')

# Renombrar columnas para display
df_display = df[['map50', 'map50_95', 'precision', 'recall', 'mean_ms', 'fps', 'params_M', 'flops_G']].copy()
df_display.columns = ['mAP50', 'mAP50-95', 'Precision', 'Recall', 'Latencia (ms)', 'FPS', 'Params (M)', 'FLOPs (G)']

print("=" * 70)
print("           COMPARACIÓN DE MODELOS – Construction PPE Detection")
print("=" * 70)
print(df_display.to_string())
print("\n* Latencia medida en CPU con imagen 416×416 (promedio de 20 runs)")

## 2. Visualización de Métricas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
models = list(eval_results.keys())
x = np.arange(len(models))
colors = ['#2196F3', '#FF5722', '#4CAF50']

metrics_to_plot = [
    ('map50',    'mAP@50',    axes[0,0]),
    ('map50_95', 'mAP@50-95', axes[0,1]),
    ('precision','Precision',  axes[1,0]),
    ('recall',   'Recall',     axes[1,1]),
]

for metric_key, metric_label, ax in metrics_to_plot:
    values = [eval_results[m].get(metric_key, 0) for m in models]
    bars = ax.bar(models, values, color=colors[:len(models)], edgecolor='white', linewidth=1.5)
    ax.set_title(metric_label, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Score")
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, label='0.5 baseline')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle("Comparación de Métricas de Detección – Construction-PPE Dataset", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(DATA_DIR / "comparison_metrics.png", bbox_inches='tight', dpi=150)
plt.show()

## 3. Trade-off Velocidad vs. Precisión

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for i, model_name in enumerate(models):
    if model_name not in latency_results:
        continue
    fps  = latency_results[model_name]['fps']
    map50 = eval_results[model_name]['map50']
    params = model_info.get(model_name, {}).get('params_M', 5)
    
    sc = ax.scatter(fps, map50, s=params * 20, color=colors[i], 
                    alpha=0.85, edgecolors='black', linewidth=1.5, zorder=5)
    ax.annotate(f"{model_name}\n({params}M params)",
                xy=(fps, map50), xytext=(10, 5), textcoords='offset points',
                fontsize=10, fontweight='bold', color=colors[i])

ax.set_xlabel("FPS (frames por segundo) – mayor es mejor", fontsize=12)
ax.set_ylabel("mAP@50 – mayor es mejor", fontsize=12)
ax.set_title("Trade-off: Velocidad vs. Precisión\n(tamaño del punto = nº de parámetros)", fontsize=13)
ax.grid(True, alpha=0.3)

# Zona de despliegue práctico (>15 FPS, >0.6 mAP)
ax.axvline(x=15, color='orange', linestyle='--', alpha=0.5, label='15 FPS mínimo')
ax.axhline(y=0.6, color='green', linestyle='--', alpha=0.5, label='0.6 mAP50 objetivo')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(DATA_DIR / "speed_accuracy_tradeoff.png", bbox_inches='tight', dpi=150)
plt.show()

## 4. Curvas de Entrenamiento

In [ ]:
import csv

run_dirs = {
    'YOLOv8n':  RUNS_DIR / 'yolov8n_ppe',
    'YOLOv8m':  RUNS_DIR / 'yolov8m_ppe',
    'YOLOv11n': RUNS_DIR / 'yolov11n_ppe',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_metrics = ['metrics/mAP50(B)', 'train/box_loss', 'val/box_loss']
plot_labels  = ['mAP@50 (val)', 'Box Loss (train)', 'Box Loss (val)']
plot_colors  = {'YOLOv8n': '#2196F3', 'YOLOv8m': '#FF5722', 'YOLOv11n': '#4CAF50'}

for ax, metric, label in zip(axes, plot_metrics, plot_labels):
    for model_name, run_dir in run_dirs.items():
        results_csv = run_dir / 'results.csv'
        if not results_csv.exists():
            print(f"⚠️  No encontrado: {results_csv}")
            continue
        df_run = pd.read_csv(results_csv)
        df_run.columns = [c.strip() for c in df_run.columns]  # strip whitespace
        if metric in df_run.columns:
            ax.plot(df_run['epoch'], df_run[metric],
                    label=model_name, color=plot_colors[model_name], linewidth=2)
    ax.set_xlabel("Época")
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Curvas de Entrenamiento por Modelo", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(DATA_DIR / "training_curves.png", bbox_inches='tight', dpi=150)
plt.show()

## 5. Métricas por Clase – Mejor Modelo

In [ ]:
import torch

# Elegir el mejor modelo según mAP50
best_model_name = max(eval_results, key=lambda m: eval_results[m].get('map50', 0))
best_model_path = MODELS_DIR / f"{best_model_name.lower().replace('v','v')}_best.pt"

print(f"Mejor modelo: {best_model_name} (mAP50 = {eval_results[best_model_name]['map50']:.4f})")

# Re-evaluar con métricas por clase
model_best = YOLO(str(best_model_path))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

metrics_per_class = model_best.val(
    data=DATASET_YAML,
    device=device,
    workers=0,
    verbose=True,
)

In [ ]:
# Extraer métricas por clase
class_names = ds_config['class_names']

try:
    ap_per_class = metrics_per_class.box.ap50   # array [n_classes]
    
    df_per_class = pd.DataFrame({
        'Clase': class_names[:len(ap_per_class)],
        'AP@50': np.round(ap_per_class, 4),
    }).sort_values('AP@50', ascending=True)

    # Color: verde si EPP presente, rojo si EPP ausente
    def class_color(cls):
        if cls.startswith('NO-'):
            return '#FF5722'
        elif cls in ['Hardhat', 'Mask', 'Safety Vest']:
            return '#4CAF50'
        return '#90A4AE'

    bar_colors = [class_color(c) for c in df_per_class['Clase']]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(df_per_class['Clase'], df_per_class['AP@50'],
                   color=bar_colors, edgecolor='white', linewidth=1)

    for bar, val in zip(bars, df_per_class['AP@50']):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

    legend_patches = [
        mpatches.Patch(color='#4CAF50', label='EPP Presente'),
        mpatches.Patch(color='#FF5722', label='EPP Ausente (incumplimiento)'),
        mpatches.Patch(color='#90A4AE', label='Otros'),
    ]
    ax.legend(handles=legend_patches, loc='lower right', fontsize=10)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Average Precision @ 50 IoU")
    ax.set_title(f"AP@50 por Clase – {best_model_name}", fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(DATA_DIR / "ap_per_class.png", bbox_inches='tight', dpi=150)
    plt.show()
except Exception as e:
    print(f"No se pudo extraer AP por clase: {e}")

## 6. Visualización de Predicciones del Mejor Modelo

In [ ]:
import cv2
import random
from PIL import Image

dataset_path = Path(DATASET_YAML).parent
test_images = list((dataset_path / 'valid' / 'images').glob('*.jpg'))
random.seed(2024)
samples = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    results = model_best.predict(str(img_path), conf=0.4, verbose=False, device=device)
    annotated = results[0].plot()  # RGB con boxes dibujadas
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    ax.axis('off')
    n_dets = len(results[0].boxes)
    ax.set_title(f"{img_path.name[:25]}\n{n_dets} detecciones", fontsize=8)

plt.suptitle(f"Predicciones del Mejor Modelo ({best_model_name}) – conf ≥ 0.4", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(DATA_DIR / "best_model_predictions.png", bbox_inches='tight', dpi=150)
plt.show()

## 7. Análisis y Justificación del Modelo Óptimo

In [ ]:
print("=" * 65)
print("  ANÁLISIS COMPARATIVO – Sistema de Monitoreo EPP")
print("=" * 65)

for model_name in eval_results:
    e = eval_results[model_name]
    l = latency_results.get(model_name, {})
    params = model_info.get(model_name, {}).get('params_M', '?')
    
    print(f"\n📦 {model_name}")
    print(f"   Parámetros:   {params}M")
    print(f"   mAP50:        {e['map50']:.4f}")
    print(f"   mAP50-95:     {e['map50_95']:.4f}")
    print(f"   Precision:    {e['precision']:.4f}")
    print(f"   Recall:       {e['recall']:.4f}")
    print(f"   Latencia:     {l.get('mean_ms','?')} ms ({l.get('fps','?')} FPS)")

print()
print("=" * 65)
print("  CONCLUSIÓN")
print("=" * 65)
print(f"""
Para un sistema de monitoreo de EPP en tiempo real:

• YOLOv8n: El más rápido (menor latencia). Ideal para edge devices
  o cuando la velocidad es crítica. Trade-off: menor mAP.

• YOLOv8m: El más preciso entre los v8, pero el más lento.
  Recomendado si se dispone de GPU y se prioriza precisión.

• YOLOv11n: Nueva arquitectura con menos parámetros que v8n
  pero arquitectura mejorada. Mejor balance speed/accuracy en 
  arquitecturas recientes.

→ Modelo seleccionado para la demo: {best_model_name}
  (mayor mAP50 entre los entrenados)
""")

---
## ✅ Checklist Notebook 3
- [ ] Tabla comparativa de métricas generada
- [ ] Gráfico de trade-off velocidad/precisión generado
- [ ] Curvas de entrenamiento comparadas
- [ ] AP por clase del mejor modelo analizado
- [ ] Predicciones visualizadas
- [ ] Modelo óptimo justificado

**Próximo paso:** Notebook 4 – Tracking con ByteTrack y Demo